In [2]:
import os
import sys
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, TimeoutError as FutureTimeoutError, as_completed

# 获取项目根目录（Lab 的父目录）
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
sys.path.insert(0, project_root)
lib_path = os.path.join(project_root, 'Lib')
sys.path.insert(0, lib_path)

print(f"项目根目录: {project_root}")
print(f"Lib 目录存在: {os.path.exists(os.path.join(project_root, 'Lib'))}")

项目根目录: c:\Users\JoeyC\Desktop\Find_Job_Pipe_Line_V2
Lib 目录存在: True


In [20]:
from Lib.json_yaml_IO import *
from Lib.Html_Analist import HtmlAnalist
from Lib.Batch_Run import BatchRun
from Lib.LLM_Analysis import LLM_model

In [21]:
from minio import Minio
from minio.error import S3Error

client = Minio(
    endpoint="localhost:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)

In [22]:
from Lib.exe_SQL import Exe_SQL
import pandas as pd
import numpy as np

In [23]:
duckdb_engine = Exe_SQL()
duckdb_engine.connect()

In [24]:
df = duckdb_engine.execute_sql_file('read_co_info.sql')

In [25]:
df.head()

,company_name,company_url,company_box_content
0,英伟达,https://www.linkedin.com/company/nvidia/about,"公司简介;英伟达;4,268,977 位关注者;关注;计算机硬件制造业;10,001+ 人;..."
1,阿西布朗勃法瑞公司（ABB）,https://www.linkedin.com/company/abb/about,"公司简介;阿西布朗勃法瑞公司（ABB）;3,959,743 位关注者;关注;自动化机械制造业..."
2,Crypto.com,https://www.linkedin.com/company/cryptocom/about,"公司简介;Crypto.com;718,034 位关注者;关注;金融服务;1,001-5,0..."
3,Selby Jennings,https://www.linkedin.com/company/selby-jenning...,"公司简介;Selby Jennings;989,649 位关注者;关注;专业服务;1,001..."
4,WorldQuant,https://www.linkedin.com/company/worldquant/about,"公司简介;WorldQuant;143,218 位关注者;关注;金融服务;501-1,000..."


In [26]:
prompts_path = os.path.join(project_root, 'Prompts/company_judgement.yaml')
prompts_config = read_yaml(prompts_path)
prompt = prompts_config.get('Company_score_prompt')

In [27]:
llm_model = LLM_model(
    api_key="sk-113e8f8fe0ad4e73b73a4232ab5b4d36",
    base_url="https://api.deepseek.com",
    prompts_file=prompts_path
)

In [28]:
import pandas as pd
import json
test_df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
def test_function(a, b):
    json_result = {
        'c': int(a + b),  # 转换为原生 Python int
        'd': int(a - b)   # 转换为原生 Python int
    }
    # 使用 json.dumps 而不是 str() 来创建有效的 JSON 字符串
    json_text = json.dumps(json_result)
    return json_text

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {}

    # 使用索引而不是字符串作为 key
    test_index = 0
    for idx, row in test_df.iterrows():
        a = row['a']
        b = row['b']
        future = executor.submit(test_function, a, b)
        futures[future] = test_index
        test_index += 1  # 修复：需要递增索引

    for future in as_completed(futures):
        try:
            result = future.result()
            result = json.loads(result)
            # 正确的方式：从字典中提取值并赋值给 DataFrame
            test_df.loc[futures[future], 'c'] = result['c']
            test_df.loc[futures[future], 'd'] = result['d']
            print(f"Index: {futures[future]}")
            print(f"Result: {result}")
        except Exception as e:
            print(f"Error: {e}")

print(test_df)

Index: 1
Result: {'c': 7, 'd': -3}
Index: 0
Result: {'c': 5, 'd': -3}
Index: 2
Result: {'c': 9, 'd': -3}
   a  b    c    d
0  1  4  5.0 -3.0
1  2  5  7.0 -3.0
2  3  6  9.0 -3.0


In [29]:
import json
def get_company_score(content):
    messages = llm_model.set_messages(
        user_input=content,
        system_prompt=prompt
    )
    result = llm_model.get_json_response(
        messages=messages
    )
    return result

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {}

    # 使用 enumerate 或 iterrows 来获取索引
    for idx, row in df.iterrows():
        # 处理 company_box_content（假设它是一个列表或类似结构）
        if isinstance(row['company_box_content'], list):
            content = [item.strip() if isinstance(item, str) else str(item).strip() for item in row['company_box_content']]
        else:
            # 如果不是列表，尝试转换为列表
            content = [str(row['company_box_content']).strip()]

        future = executor.submit(get_company_score, content)
        futures[future] = idx  # 使用 DataFrame 索引作为 key

    for future in as_completed(futures):
        try:
            result = future.result()
            result = json.loads(result)
            
            # 正确的方式：从字典中提取每个值并赋值
            idx = futures[future]
            df.loc[idx, 'company_size'] = result.get('company_size', 0)
            df.loc[idx, 'ownership'] = result.get('ownership', 0)
            df.loc[idx, 'industry'] = result.get('industry', 0)
            df.loc[idx, 'mentions'] = result.get('mentions', 0)
            df.loc[idx, 'culture'] = result.get('culture', 0)
            df.loc[idx, 'nationality'] = result.get('nationality', 0)

            print(f"Index: {futures[future]}")
            
        except Exception as e:
            print(f"Error at index {futures[future]}: {e}")


Index: 0
Index: 2
Index: 4
Index: 3
Index: 1
Index: 5
Index: 6
Index: 9
Index: 8
Index: 7
Index: 11
Index: 10
Index: 13
Index: 12
Index: 18
Index: 14
Index: 17
Index: 19
Index: 16
Index: 15
Index: 20
Index: 21
Index: 24
Index: 23
Index: 22
Index: 25
Index: 26
Index: 27
Index: 28
Index: 30
Index: 31
Index: 32
Index: 33
Index: 34
Index: 35
Index: 37
Index: 29
Index: 36
Index: 38
Index: 39
Index: 40
Index: 43
Index: 44
Index: 41
Index: 42
Index: 46
Index: 45
Index: 48
Index: 47
Index: 51
Index: 50
Index: 49
Index: 53
Index: 52
Index: 54
Index: 55
Index: 58
Index: 57
Index: 56
Index: 60
Index: 59
Index: 63
Index: 62
Index: 61
Index: 64
Index: 65
Index: 67
Index: 66
Index: 68
Index: 69
Index: 70
Index: 71
Index: 73
Index: 72
Index: 74
Index: 77
Index: 75
Index: 76
Index: 78
Index: 83
Index: 79
Index: 82
Index: 81
Index: 85
Index: 80
Index: 84
Index: 86
Index: 88
Index: 87
Index: 92
Index: 89
Index: 90
Index: 93
Index: 91
Index: 94
Index: 96
Index: 95
Index: 97
Index: 98
Index: 100
Index: 10

In [ ]:
df['company_score'] = 3 * df['company_size'] +  2 * df['ownership'] + 2 * df['industry'] + 1.5 * df['mentions'] + 0.05 * df['culture'] + df['nationality']

In [41]:
for company in df[df['company_score'] >= 0]['company_name']:
    print(company)

联合服务（香港）有限公司
多准数据
浙江深大智能科技有限公司
诺得物流
睿和
Sharp Brains
Flexport
北京万古恒信科技有限公司
北京瑞天欣实数据科技有限公司
深圳市睿服科技有限公司
ArcherMind Technology (Nanjing) Co., Ltd.
杭州推宝科技有限公司
高知特智能技术（上海）有限公司
苏州挚途科技有限公司
Counterpoint Research
成都市易冲半导体有限公司
成都天府软件园有限公司
Huali
四维图新
北京爱科迪通信技术股份有限公司
台成机电
维京游轮
Belimo
Shanghai Wondertek Software. Co., Ltd.(网达软件)
青岛文达通科技股份有限公司
郑州森储软件科技有限公司
智控
深圳明锐理想科技有限公司
AEC LIGHTING SOLUTIONS CO., LTD
Wati
Nilfisk
科技有限公司


In [42]:
import pyarrow as pa
import pyarrow.parquet as pq
from io import BytesIO

In [43]:
def df_to_parquet(df):
    buffer = BytesIO()
    table = pa.Table.from_pandas(df)
    pq.write_table(table, buffer, compression='snappy')
    buffer.seek(0)
    return buffer.getvalue() 

In [44]:
buffer = df_to_parquet(df)

In [ ]:
from datetime import datetime
def upload_to_minio(df):
    today = datetime.now().strftime('%Y-%m-%d')
    buffer_bytes = df_to_parquet(df)
    # Wrap bytes in BytesIO to provide file-like interface
    buffer = BytesIO(buffer_bytes)
    client.put_object(
        bucket_name='jobdatabucket',
        object_name='bronze/company_info/dt=' + today + '/company_info_with_score.parquet',
        data=buffer,
        length=len(buffer_bytes)
    )
upload_to_minio(df)